# 2. Training-only EDA and seasonality

Choose modelling settings from the fit period. Annual and weekly seasonal models are not silently assumed.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
from telco_anomaly.pipeline import runtime

POLICY = runtime(ROOT / "configs/pipeline.yml")
PACK = Path(POLICY["pack"])
RUN = Path(POLICY["run"])


In [ ]:
from telco_anomaly.pipeline import observations
from telco_anomaly.synthetic_pipeline import split_boundaries
from telco_anomaly.features import fit_seasonality
import matplotlib.pyplot as plt

grid, manifest = observations(POLICY)
train_end = split_boundaries(manifest)["train"][1]
training = grid.loc[grid.timestamp_utc < train_end]
decisions, evidence = fit_seasonality(training, train_end, POLICY["timezone"])
display(evidence)
print("Approved daily metrics:", decisions["approved_daily_metrics"])
display(training.groupby("gap_kind").size().rename("rows"))
display(training.select_dtypes("number").describe().T)

In [ ]:
entity = training.ont_id.iloc[0]
trace = training.loc[training.ont_id == entity].set_index("timestamp_utc")
trace[["rx_power_dbm", "olt_rx_power_dbm", "temperature_c"]].plot(
    subplots=True, figsize=(12, 7), title=entity
)
plt.tight_layout()

Missing rows remain missing; the grid exposes silence. Gap classification at the first returning observation uses elapsed time, uptime and reboot count. Daily profiles require repeated-day support and are frozen during fitting. Noise and fault distributions remain uncalibrated assumptions. These plots are diagnostics, not proof of field realism.